# Importing and installing libraries and packages

In [1]:
import pandas as pd
import numpy as np

df=pd.read_csv('truedata.csv')

In [2]:
df = df.dropna()

In [3]:
!pip install tensorflow
!pip install keras

In [5]:
df

,Strain Rate,Temperature,True Strain,True Stress,True Plastic Strain
0,0.0001,27.0,0.10683,939.31764,0.00030
1,0.0001,27.0,0.10683,939.32333,0.00030
2,0.0001,27.0,0.10684,939.33680,0.00031
3,0.0001,27.0,0.10684,939.34085,0.00031
4,0.0001,27.0,0.10685,939.34472,0.00032
...,...,...,...,...,...
162754,0.0100,500.0,0.06196,597.07372,0.01804
162755,0.0100,500.0,0.06218,597.09635,0.01826
162756,0.0100,500.0,0.06239,597.11009,0.01847
162757,0.0100,500.0,0.06260,597.11561,0.01868


# Standardization, splitting the data 

In [6]:
TargetVariable=['True Stress']
Predictors=['Strain Rate','Temperature','True Plastic Strain']
 
X=df[Predictors].values
y=df[TargetVariable].values
 
### Sandardization of data ###
from sklearn.preprocessing import StandardScaler
PredictorScaler=StandardScaler()
TargetVarScaler=StandardScaler()
 
# Storing the fit object for later reference
PredictorScalerFit=PredictorScaler.fit(X)
TargetVarScalerFit=TargetVarScaler.fit(y)
 
# Generating the standardized values of X and y
X=PredictorScalerFit.transform(X)
y=TargetVarScalerFit.transform(y)
 
# Split the data into training and testing set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
 
# Quick sanity check with the shapes of Training and testing datasets
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(113931, 3)
(113931, 1)
(48828, 3)
(48828, 1)


# Hyperparameter tuning and cross validation

In [12]:
# Function to generate Deep ANN model 
def make_regression_ann(learning_rate,dropout_rate):
    from keras.models import Sequential
    from keras.layers import Dense
    
    model = Sequential()
    model.add(Dense(units=5, input_dim=3, kernel_initializer='normal', activation='relu'))
    model.add(Dense(units=5, kernel_initializer='normal', activation='relu'))
    model.add(Dense(1, kernel_initializer='normal'))
    model.compile(loss='mean_squared_error', optimizer='adam')
    return model
 
###########################################
from sklearn.model_selection import GridSearchCV
from keras.wrappers.scikit_learn import KerasRegressor
 
# Listing all the parameters to try
Parameter_Trials={'batch_size':[20],
                  'epochs':[30],'learning_rate':[0.001,0.01,0.1],
                  'dropout_rate': [0.005,0.1,0.2]
                 }

filepath  = 'C:/Users/ankit/Downloads/Files for HPC'
from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor = 'val_loss', patience=5)
from keras.callbacks import ModelCheckpoint
checkpoint = ModelCheckpoint(filepath, monitor='val_loss',mode='min',save_best_only=True,verbose=1)

 
# Creating the regression ANN model
RegModel=KerasRegressor(make_regression_ann, validation_split=0.25, verbose=0, callbacks=[early_stopping, checkpoint])
 
###########################################
from sklearn.metrics import make_scorer
 
# Defining a custom function to calculate accuracy
def Accuracy_Score(orig,pred):
    MAPE = np.mean(100 * (np.abs(orig-pred)/orig))
    print('#'*70,'Accuracy:', 100-MAPE)
    return(100-MAPE)
 
custom_Scoring=make_scorer(Accuracy_Score, greater_is_better=True)
 
#########################################
# Creating the Grid search space
# See different scoring methods by using sklearn.metrics.SCORERS.keys()
grid_search=GridSearchCV(estimator=RegModel, 
                         param_grid=Parameter_Trials, 
                         scoring=custom_Scoring, 
                         cv=10,n_jobs=-2)
 
#########################################
# Measuring how much time it took to find the best params
import time
StartTime=time.time()
 
# Running Grid Search for different paramenters
grid_search.fit(X,y, verbose=1)
 
EndTime=time.time()
print("########## Total Time Taken: ", round((EndTime-StartTime)/60), 'Minutes')
 
print('### Printing Best parameters ###')
grid_search.best_params_

C:\Users\ankit\AppData\Local\Temp\ipykernel_19616\615018142.py:31: DeprecationWarning: KerasRegressor is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  RegModel=KerasRegressor(make_regression_ann, validation_split=0.25, verbose=0, callbacks=[early_stopping, checkpoint])
C:\Users\ankit\Downloads\Anaconda\lib\site-packages\sklearn\model_selection\_validation.py:372: FitFailedWarning: 
14 fits failed out of a total of 90.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\ankit\Downloads\Anaconda\lib\site-packages\sklearn\model_selection\_validat

Epoch 1/30
6073/6104 [============================>.] - ETA: 0s - loss: 0.0823
Epoch 1: val_loss improved from inf to 1.64021, saving model to C:/Users/ankit/Downloads\Files for HPC


INFO:tensorflow:Assets written to: C:/Users/ankit/Downloads\Files for HPC\assets


INFO:tensorflow:Assets written to: C:/Users/ankit/Downloads\Files for HPC\assets


6104/6104 [==============================] - 6s 886us/step - loss: 0.0820 - val_loss: 1.6402
Epoch 2/30
6099/6104 [============================>.] - ETA: 0s - loss: 0.0373
Epoch 2: val_loss did not improve from 1.64021
6104/6104 [==============================] - 5s 774us/step - loss: 0.0373 - val_loss: 1.6417
Epoch 3/30
6097/6104 [============================>.] - ETA: 0s - loss: 0.0373
Epoch 3: val_loss improved from 1.64021 to 1.63790, saving model to C:/Users/ankit/Downloads\Files for HPC


...

6104/6104 [==============================] - 5s 845us/step - loss: 0.0372 - val_loss: 1.5733
Epoch 21/30
6023/6104 [============================>.] - ETA: 0s - loss: 0.0372
Epoch 21: val_loss improved from 1.57325 to 1.57130, saving model to C:/Users/ankit/Downloads\Files for HPC


INFO:tensorflow:Assets written to: C:/Users/ankit/Downloads\Files for HPC\assets


INFO:tensorflow:Assets written to: C:/Users/ankit/Downloads\Files for HPC\assets


6104/6104 [==============================] - 5s 828us/step - loss: 0.0372 - val_loss: 1.5713
Epoch 22/30
6057/6104 [============================>.] - ETA: 0s - loss: 0.0371
Epoch 22: val_loss did not improve from 1.57130
6104/6104 [==============================] - 5s 778us/step - loss: 0.0372 - val_loss: 1.5742
Epoch 23/30
6078/6104 [============================>.] - ETA: 0s - loss: 0.0372
Epoch 23: val_loss did not improve from 1.57130
6104/6104 [==============================] - 5s 808us/step - loss: 0.0372 - val_loss: 1.5767
Epoch 24/30
6089/6104 [============================>.] - ETA: 0s - loss: 0.0372
Epoch 24: val_loss did not improve from 1.57130
6104/6104 [==============================] - 5s 782us/step - loss: 0.0372 - val_loss: 1.5893
Epoch 25/30
6027/6104 [============================>.] - ETA: 0s - loss: 0.0371
Epoch 25: val_loss did not improve from 1.57130
6104/6104 [==============================] - 5s 772us/step - loss: 0.0372 - val_loss: 1.5885
Epoch 26/30
6049/6104 [

{'batch_size': 20, 'dropout_rate': 0.1, 'epochs': 30, 'learning_rate': 0.001}

# Fitting the model again with the best parameters, generate predictions and evaluating the model

In [15]:

# importing the libraries
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout

filepath  = 'C:/Users/ankit/Downloads/Files for HPC'
from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor = 'val_loss', patience=5)
from keras.callbacks import ModelCheckpoint
checkpoint = ModelCheckpoint(filepath, monitor='val_loss',mode='min',save_best_only=True,verbose=1)

# create ANN model
model = Sequential()
 
# Defining the Input layer and FIRST hidden layer, both are same!
model.add(Dense(units=5, input_dim=3, kernel_initializer='normal', activation='relu'))
model.add(Dropout(0.1))
# Defining the Second layer of the model
# after the first layer we don't have to specify input_dim as keras configure it automatically
model.add(Dense(units=5, kernel_initializer='normal', activation='relu'))
model.add(Dropout(0.1))
# The output neuron is a single fully connected node 
# Since we will be predicting a single number
model.add(Dense(1, kernel_initializer='normal'))
 
# Compiling the model
model.compile(loss='mean_squared_error', optimizer=keras.optimizers.Adam(learning_rate = 0.001))
 
# Fitting the ANN to the Training set
model.fit(X_train, y_train ,batch_size = 20, epochs = 30, validation_split=0.25, verbose=0, callbacks=[early_stopping, checkpoint])
 
# Generating Predictions on testing data
Predictions=model.predict(X_test)
 
# Scaling the predicted Price data back to original price scale
Predictions=TargetVarScalerFit.inverse_transform(Predictions)
 
# Scaling the y_test Price data back to original price scale
y_test_orig=TargetVarScalerFit.inverse_transform(y_test)
 
# Scaling the test data back to original scale
Test_Data=PredictorScalerFit.inverse_transform(X_test)
 
TestingData=pd.DataFrame(data=Test_Data, columns=Predictors)
TestingData['Stress']=y_test_orig
TestingData['Predicted Stress']=Predictions
TestingData


Epoch 1: val_loss improved from inf to 0.05199, saving model to C:/Users/ankit/Downloads\Files for HPC


INFO:tensorflow:Assets written to: C:/Users/ankit/Downloads\Files for HPC\assets


INFO:tensorflow:Assets written to: C:/Users/ankit/Downloads\Files for HPC\assets



Epoch 2: val_loss improved from 0.05199 to 0.04412, saving model to C:/Users/ankit/Downloads\Files for HPC


INFO:tensorflow:Assets written to: C:/Users/ankit/Downloads\Files for HPC\assets


...


Epoch 12: val_loss improved from 0.03696 to 0.03619, saving model to C:/Users/ankit/Downloads\Files for HPC


INFO:tensorflow:Assets written to: C:/Users/ankit/Downloads\Files for HPC\assets


INFO:tensorflow:Assets written to: C:/Users/ankit/Downloads\Files for HPC\assets



Epoch 13: val_loss did not improve from 0.03619

Epoch 14: val_loss did not improve from 0.03619

Epoch 15: val_loss did not improve from 0.03619

Epoch 16: val_loss did not improve from 0.03619

Epoch 17: val_loss did not improve from 0.03619
1526/1526 [==============================] - 1s 488us/step


,Strain Rate,Temperature,True Plastic Strain,Stress,Predicted Stress
0,0.0001,100.0,0.00892,781.83656,767.241699
1,0.0001,300.0,0.01215,622.26907,672.584106
2,0.0001,200.0,0.02006,694.36165,712.741333
3,0.0001,500.0,0.01024,790.93119,817.324402
4,0.0001,400.0,0.02834,731.18290,739.924988
...,...,...,...,...,...
48823,0.0001,500.0,0.03309,917.60527,897.899963
48824,0.0001,100.0,0.00278,777.83329,754.005859
48825,0.0001,400.0,0.04224,740.82561,773.647217
48826,0.0010,500.0,0.11835,1152.24326,1117.856445


In [18]:
# Computing the absolute percent error
APE=100*(abs(TestingData['Stress']-TestingData['Predicted Stress'])/TestingData['Stress'])
TestingData['APE']=APE
 
print('The Accuracy of ANN model is:', np.mean(APE))
TestingData

The Accuracy of ANN model is: 2.9844653926677136


,Strain Rate,Temperature,True Plastic Strain,Stress,Predicted Stress,APE
0,0.0001,100.0,0.00892,781.83656,767.241699,1.866741
1,0.0001,300.0,0.01215,622.26907,672.584106,8.085736
2,0.0001,200.0,0.02006,694.36165,712.741333,2.646990
3,0.0001,500.0,0.01024,790.93119,817.324402,3.336980
4,0.0001,400.0,0.02834,731.18290,739.924988,1.195609
...,...,...,...,...,...,...
48823,0.0001,500.0,0.03309,917.60527,897.899963,2.147471
48824,0.0001,100.0,0.00278,777.83329,754.005859,3.063308
48825,0.0001,400.0,0.04224,740.82561,773.647217,4.430409
48826,0.0010,500.0,0.11835,1152.24326,1117.856445,2.984336


In [19]:
# Computing the absolute percent error
APE=(abs(TestingData['Stress']-TestingData['Predicted Stress']))
TestingData['APE']=APE
 
print('The Accuracy of ANN model is:', np.mean(APE))
TestingData

The Accuracy of ANN model is: 23.282291048498003


,Strain Rate,Temperature,True Plastic Strain,Stress,Predicted Stress,APE
0,0.0001,100.0,0.00892,781.83656,767.241699,14.594861
1,0.0001,300.0,0.01215,622.26907,672.584106,50.315036
2,0.0001,200.0,0.02006,694.36165,712.741333,18.379683
3,0.0001,500.0,0.01024,790.93119,817.324402,26.393212
4,0.0001,400.0,0.02834,731.18290,739.924988,8.742088
...,...,...,...,...,...,...
48823,0.0001,500.0,0.03309,917.60527,897.899963,19.705307
48824,0.0001,100.0,0.00278,777.83329,754.005859,23.827431
48825,0.0001,400.0,0.04224,740.82561,773.647217,32.821607
48826,0.0010,500.0,0.11835,1152.24326,1117.856445,34.386815


In [20]:
import statistics
var = (statistics.variance(TestingData['Predicted Stress']))
print(var)
chi_sq = np.sum(((TestingData['Stress']-TestingData['Predicted Stress'])**2)/var)
red_chi_sq = chi_sq/5074
print('The reduced chi squared value for RFR is', red_chi_sq)

16944.086303533415
The reduced chi squared value for RFR is 0.5342657712656879


# Validating the model

In [21]:
dfvalid=pd.read_csv('150Cvalidationdata.csv')

In [22]:
dfvalid

,Strain Rate,Temperature,True Strain,True Stress,True Plastic Strain,Ridge Predicted Stress,RFR Predicted Stress,XGB Predicted Stress,SVR Predicted Stress
0,0.001,150,0.008686,649.481445,0.000062,633.142806,777.754361,624.89720,617.140417
1,0.001,150,0.008677,649.420837,0.000053,633.108214,777.754361,624.89720,617.104570
2,0.001,150,0.008694,649.400147,0.000070,633.172487,777.754361,624.89720,617.171199
3,0.001,150,0.008714,649.403870,0.000090,633.248762,777.754361,624.89720,617.250295
4,0.001,150,0.008722,649.421997,0.000097,633.276078,777.754361,624.89720,617.278739
...,...,...,...,...,...,...,...,...,...
5072,0.001,150,0.103290,700.204407,0.094666,988.512448,837.707106,756.38920,985.754788
5073,0.001,150,0.103298,700.248047,0.094674,988.542871,837.708480,756.38920,985.786345
5074,0.001,150,0.103348,700.315308,0.094723,988.728200,837.732867,756.38920,985.978581
5075,0.001,150,0.103320,700.301147,0.094695,988.623111,837.706032,756.38920,985.869576


In [23]:
Predictors=['Strain Rate','Temperature','True Plastic Strain']
X_valid=dfvalid[Predictors].values

In [24]:
X_valid

array([[1.0000000e-03, 1.5000000e+02, 6.1900000e-05],
       [1.0000000e-03, 1.5000000e+02, 5.2700000e-05],
       [1.0000000e-03, 1.5000000e+02, 6.9800000e-05],
       ...,
       [1.0000000e-03, 1.5000000e+02, 9.4723457e-02],
       [1.0000000e-03, 1.5000000e+02, 9.4695481e-02],
       [1.0000000e-03, 1.5000000e+02, 9.4743924e-02]])

In [25]:
from sklearn.preprocessing import StandardScaler
PredictorScaler=StandardScaler()
TargetVarScaler=StandardScaler()
 
# Storing the fit object for later reference
PredictorScalerFit=PredictorScaler.fit(X_valid)

# Generating the standardized values of X and y
X_valid=PredictorScalerFit.transform(X_valid)

In [26]:
X_valid

array([[-2.16840434e-19,  0.00000000e+00, -1.75357897e+00],
       [-2.16840434e-19,  0.00000000e+00, -1.75391575e+00],
       [-2.16840434e-19,  0.00000000e+00, -1.75328979e+00],
       ...,
       [-2.16840434e-19,  0.00000000e+00,  1.71155471e+00],
       [-2.16840434e-19,  0.00000000e+00,  1.71053064e+00],
       [-2.16840434e-19,  0.00000000e+00,  1.71230392e+00]])

In [27]:
y_pred_valid = model.predict(X_valid)

159/159 [==============================] - 0s 509us/step


In [28]:
y_pred_valid

array([[-1.2759032 ],
       [-1.2760446 ],
       [-1.2757816 ],
       ...,
       [-0.37425947],
       [-0.37425947],
       [-0.37425947]], dtype=float32)

In [29]:
y_pred_valid = TargetVarScalerFit.inverse_transform(y_pred_valid)

In [31]:
y_pred_valid

array([[619.0699],
       [619.0484],
       [619.0884],
       ...,
       [756.0348],
       [756.0348],
       [756.0348]], dtype=float32)

In [32]:
import pandas as pd 
pd.DataFrame(y_pred_valid).to_csv("annvalidationpredicteddata.csv")